# Stage 1 — Build Research Taxonomy

建立固定的三層研究領域階層（L1 / L2 / L3），輸出 `data/network-analysis/research_taxonomy.csv`。

- **L3** = `research_interests_standardized_en` 的 170 個 canonical 值本身。
- **L2** = 中等粒度的研究主題群（約 46 個）。
- **L1** = 粗粒度領域（16 個），刻意避免 `Computer Science`、`Economics` 這類全系涵蓋的類別。

此表為**固定資產**：Stage 2~4 一律讀取，不動態推斷。

In [1]:
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "data-preprocessing").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

FACULTY_JSON = ROOT / "data" / "data-preprocessing" / "faculty_standardized.json"
OUT_DIR = ROOT / "data" / "network-analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("project root :", ROOT)
print("faculty json :", FACULTY_JSON.exists())

project root : <project-root>
faculty json : True


In [2]:
with FACULTY_JSON.open(encoding="utf-8") as fh:
    FACULTY = json.load(fh)

keyword_counts = Counter()
for rec in FACULTY:
    for kw in rec.get("research_interests_standardized_en") or []:
        keyword_counts[kw] += 1

N_FACULTY = len(FACULTY)
N_WITH_KEYWORDS = sum(1 for r in FACULTY if r.get("research_interests_standardized_en"))
print(f"faculty total            : {N_FACULTY}")
print(f"faculty with keywords    : {N_WITH_KEYWORDS} ({100 * N_WITH_KEYWORDS / N_FACULTY:.1f}%)")
print(f"canonical keywords (L3)  : {len(keyword_counts)}")

faculty total            : 741
faculty with keywords    : 662 (89.3%)
canonical keywords (L3)  : 170


## 三層階層定義

手工編定的 `L1 -> L2 -> [L3]` 對照表。設計原則：

1. L1 不得使用 `Computer Science` / `Economics` 這種涵蓋整個系所的類別，必須在系所**內部**仍有鑑別力。
2. 全體層級任一 L1 覆蓋率不得超過 50%（於下方驗證）。
3. 每個 L1 至少含 2 個 L2；每個 L3 恰好歸屬一個 L2。

In [3]:
TAXONOMY: dict[str, dict[str, list[str]]] = {
    "AI & Machine Learning": {
        "Machine Learning & Deep Learning": [
            "Machine Learning",
            "Deep Learning",
            "Neural Networks",
            "Reinforcement Learning",
            "Evolutionary Computation",
            "Fuzzy Systems",
            "Federated Learning",
            "Neuromorphic Computing",
        ],
        "Artificial Intelligence & Generative Models": [
            "Artificial Intelligence",
            "Generative AI",
            "Trustworthy AI",
        ],
        "Natural Language & Speech Processing": [
            "Natural Language Processing",
            "Computational Linguistics",
            "Machine Translation",
            "Speech Recognition",
        ],
    },
    "Vision, Multimedia & Human-Centered Computing": {
        "Computer Vision, Image & Signal Processing": [
            "Computer Vision",
            "Image Processing",
            "Pattern Recognition",
            "Signal Processing",
        ],
        "Graphics & Multimedia Systems": [
            "Computer Graphics",
            "Multimedia Systems",
        ],
        "Human-Computer Interaction & Immersive Media": [
            "Human-Computer Interaction",
            "Virtual Reality",
            "Augmented Reality",
            "Brain-Computer Interfaces",
            "Information Visualization",
        ],
    },
    "Data, Databases & Knowledge Discovery": {
        "Data Mining & Big Data Analytics": [
            "Data Mining",
            "Big Data Analytics",
            "Data Science",
            "Text Mining",
            "Web Mining",
            "Information Retrieval",
            "Recommender Systems",
            "Business Data Analytics",
            "Spatial Clustering",
            "Scientific Modeling",
        ],
        "Database & Storage Systems": [
            "Databases",
            "Storage Systems",
        ],
        "Social & Web Network Analytics": [
            "Social Network Analysis",
            "Social Media Analytics",
        ],
    },
    "Systems, Software & Architecture": {
        "Computer Architecture & Hardware Design": [
            "Computer Architecture",
            "Electronic Design Automation",
            "Quantum Computing",
        ],
        "Operating, Real-Time & Embedded Systems": [
            "Operating Systems",
            "Real-Time Systems",
            "Embedded Systems",
            "Cyber-Physical Systems",
            "Digital Twins",
        ],
        "Parallel & Distributed Computing": [
            "Parallel Computing",
            "High-Performance Computing",
            "Distributed Systems",
        ],
        "Software Engineering, Languages & Verification": [
            "Software Engineering",
            "Programming Languages",
            "Compilers",
            "Formal Verification",
            "Formal Methods",
            "Model Checking",
            "Software Verification",
        ],
    },
    "Networks & Communications": {
        "Computer & Wireless Networks": [
            "Computer Networks",
            "Wireless Networks",
            "Wireless Communications",
        ],
        "Mobile, IoT & Pervasive Computing": [
            "Mobile Computing",
            "Internet of Things",
            "Pervasive Computing",
            "Intelligent Transportation Systems",
        ],
        "Cloud & Edge Computing": [
            "Cloud Computing",
            "Edge Computing",
        ],
    },
    "Security, Privacy & Cryptography": {
        "Information & Systems Security": [
            "Information Security",
            "Network Security",
            "Systems Security",
            "Malware Analysis",
        ],
        "Cryptography, Blockchain & Privacy": [
            "Cryptography",
            "Blockchain",
            "Privacy-Preserving Computing",
        ],
    },
    "Theory & Algorithms": {
        "Algorithms & Computational Complexity": [
            "Algorithms",
            "Theory of Computation",
            "Computational Complexity",
        ],
        "Discrete Mathematics & Information Theory": [
            "Graph Theory",
            "Discrete Mathematics",
            "Information Theory",
            "Coding Theory",
        ],
    },
    "Robotics, Control & Smart Manufacturing": {
        "Robotics & Automation": [
            "Robotics",
            "Automation and Control",
        ],
        "Smart Manufacturing & Industrial Systems": [
            "Smart Manufacturing",
        ],
    },
    "Biomedical & Health Informatics": {
        "Bioinformatics & Systems Biology": [
            "Bioinformatics",
            "Systems Biology",
        ],
        "Medical Informatics & Imaging": [
            "Medical Informatics",
            "Medical Imaging",
            "Smart Healthcare",
        ],
        "Biostatistics & Survival Analysis": [
            "Biostatistics",
            "Survival Analysis",
        ],
    },
    "Statistics & Statistical Methods": {
        "Statistical Theory & Inference": [
            "Statistics",
            "Statistical Inference",
            "Probability Theory",
            "High-Dimensional Statistics",
            "Causal Inference",
        ],
        "Bayesian & Computational Statistics": [
            "Bayesian Statistics",
            "Statistical Computing",
        ],
        "Time Series & Change Point Analysis": [
            "Time Series Analysis",
            "Change Point Analysis",
        ],
        "Applied & Industrial Statistics": [
            "Multivariate Analysis",
            "Regression Analysis",
            "Experimental Design",
            "Survey Sampling",
            "Industrial Statistics",
            "Statistical Process Control",
            "Reliability Analysis",
            "Statistical Consulting",
        ],
    },
    "Applied Mathematics & Operations Research": {
        "Optimization & Operations Research": [
            "Optimization",
            "Operations Research",
            "Queueing Theory",
            "Data Envelopment Analysis",
        ],
        "Mathematical Analysis & Scientific Computing": [
            "Numerical Analysis",
            "Scientific Computing",
            "Partial Differential Equations",
            "Algebraic Geometry",
            "Differential Geometry",
        ],
        "Game Theory & Mechanism Design": [
            "Game Theory",
            "Mechanism Design",
        ],
    },
    "Corporate Finance & Governance": {
        "Corporate Finance & Valuation": [
            "Corporate Finance",
            "Financial Management",
            "Business Valuation",
            "Initial Public Offerings",
            "Accounting",
        ],
        "Governance & Corporate Control": [
            "Corporate Governance",
            "Mergers and Acquisitions",
            "Special Purpose Acquisition Companies",
            "Institutional Investors",
        ],
    },
    "Asset Pricing & Financial Markets": {
        "Asset Pricing & Investments": [
            "Asset Pricing",
            "Investments",
            "Portfolio Management",
            "Asset Management",
            "Financial Economics",
        ],
        "Financial Markets & Microstructure": [
            "Financial Markets",
            "Market Microstructure",
        ],
        "Behavioral & International Finance": [
            "Behavioral Finance",
            "International Finance",
            "Real Estate Finance",
        ],
    },
    "Financial Engineering & Risk": {
        "Derivatives & Financial Engineering": [
            "Financial Derivatives",
            "Financial Engineering",
            "Interest Rate Modeling",
        ],
        "Risk, Insurance & Actuarial Science": [
            "Risk Management",
            "Credit Risk",
            "Insurance",
            "Actuarial Science",
        ],
        "Banking & Financial Technology": [
            "Banking and Financial Institutions",
            "Financial Technology",
            "Cryptocurrency",
            "Sustainable Finance",
            "Personal Finance",
        ],
    },
    "Economics & Policy": {
        "Econometrics": [
            "Econometrics",
        ],
        "Macroeconomics & Monetary Economics": [
            "Macroeconomics",
            "Monetary Economics",
        ],
        "Microeconomics & Industrial Organization": [
            "Microeconomics",
            "Industrial Organization",
            "Experimental Economics",
        ],
        "Applied & Policy Economics": [
            "Labor Economics",
            "Political Economy",
            "Public Economics",
            "Development Economics",
            "International Economics",
            "Environmental Economics",
            "Health Economics",
        ],
    },
    "Information Systems & Management": {
        "Digital Business & E-Commerce": [
            "Digital Transformation",
            "E-Commerce",
            "Service Science",
        ],
        "Innovation, Entrepreneurship & Management": [
            "Innovation Management",
            "Entrepreneurship",
            "Knowledge Management",
            "Operations Management",
        ],
    },
}

n_l1 = len(TAXONOMY)
n_l2 = sum(len(v) for v in TAXONOMY.values())
n_l3 = sum(len(ks) for v in TAXONOMY.values() for ks in v.values())
print(f"L1 = {n_l1}   L2 = {n_l2}   L3 = {n_l3}")

L1 = 16   L2 = 46   L3 = 170


## `source` 欄位規則

`explicit`：該 canonical 值本身就是傘狀（umbrella）詞——教授原文直接寫出了較上層的層級，
判定方式為該 L3 名稱是其 `parent_l2` 標籤以 ` & ` / `,` 切開後的其中一個成分（大小寫、標點正規化後比對）。

`inferred`：上層由 taxonomy 推得，教授原文只寫了較細的子領域。

In [4]:
def _norm_label(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())


def _label_parts(label: str) -> set[str]:
    parts = re.split(r"\s*&\s*|\s*,\s*", label)
    return {_norm_label(p) for p in parts if p.strip()}


rows = []
for l1, l2_map in TAXONOMY.items():
    for l2, keywords in l2_map.items():
        parts = _label_parts(l2) | _label_parts(l1)
        for kw in keywords:
            explicit = _norm_label(kw) in parts
            rows.append(
                {
                    "keyword_en": kw,
                    "parent_l2": l2,
                    "parent_l1": l1,
                    "faculty_count": int(keyword_counts.get(kw, 0)),
                    "source": "explicit" if explicit else "inferred",
                    "note": (
                        "umbrella term written verbatim by faculty"
                        if explicit
                        else "parent levels assigned by curated taxonomy"
                    ),
                }
            )

taxonomy_df = pd.DataFrame(rows).sort_values(
    ["parent_l1", "parent_l2", "faculty_count", "keyword_en"],
    ascending=[True, True, False, True],
).reset_index(drop=True)

print(taxonomy_df["source"].value_counts().to_string())
taxonomy_df.head(12)

source
inferred    115
explicit     55


,keyword_en,parent_l2,parent_l1,faculty_count,source,note
0,Artificial Intelligence,Artificial Intelligence & Generative Models,AI & Machine Learning,109,explicit,umbrella term written verbatim by faculty
1,Generative AI,Artificial Intelligence & Generative Models,AI & Machine Learning,7,inferred,parent levels assigned by curated taxonomy
2,Trustworthy AI,Artificial Intelligence & Generative Models,AI & Machine Learning,4,inferred,parent levels assigned by curated taxonomy
3,Machine Learning,Machine Learning & Deep Learning,AI & Machine Learning,94,explicit,umbrella term written verbatim by faculty
4,Deep Learning,Machine Learning & Deep Learning,AI & Machine Learning,54,explicit,umbrella term written verbatim by faculty
5,Reinforcement Learning,Machine Learning & Deep Learning,AI & Machine Learning,12,inferred,parent levels assigned by curated taxonomy
6,Neural Networks,Machine Learning & Deep Learning,AI & Machine Learning,11,inferred,parent levels assigned by curated taxonomy
7,Evolutionary Computation,Machine Learning & Deep Learning,AI & Machine Learning,4,inferred,parent levels assigned by curated taxonomy
8,Fuzzy Systems,Machine Learning & Deep Learning,AI & Machine Learning,2,inferred,parent levels assigned by curated taxonomy
9,Neuromorphic Computing,Machine Learning & Deep Learning,AI & Machine Learning,2,inferred,parent levels assigned by curated taxonomy


## 驗證

In [5]:
canonical = set(keyword_counts)
mapped = set(taxonomy_df["keyword_en"])

missing = sorted(canonical - mapped)
extra = sorted(mapped - canonical)
dupes = sorted(taxonomy_df.loc[taxonomy_df.duplicated("keyword_en"), "keyword_en"])

checks: list[tuple[str, bool, str]] = []
checks.append((f"170 個 canonical 值全數對應（實際 {len(canonical)}）", not missing, f"missing={missing[:10]}"))
checks.append(("taxonomy 未含 canonical 之外的值", not extra, f"extra={extra[:10]}"))
checks.append(("每個 keyword 只出現一次", not dupes, f"duplicated={dupes[:10]}"))
checks.append(("無孤兒（parent_l2 / parent_l1 皆非空）",
               bool(taxonomy_df["parent_l2"].notna().all() and taxonomy_df["parent_l1"].notna().all()), ""))

l2_per_l1 = taxonomy_df.groupby("parent_l1")["parent_l2"].nunique()
bad_l1 = l2_per_l1[l2_per_l1 < 2]
checks.append(("每個 L1 至少含 2 個 L2", bad_l1.empty, f"{bad_l1.to_dict()}"))

# 全體層級 L1 覆蓋率（以持有該 L1 任一領域的教授人數 / 662 計）
kw2l1 = dict(zip(taxonomy_df["keyword_en"], taxonomy_df["parent_l1"]))
kw2l2 = dict(zip(taxonomy_df["keyword_en"], taxonomy_df["parent_l2"]))

l1_people: dict[str, set[str]] = {l1: set() for l1 in TAXONOMY}
for rec in FACULTY:
    kws = rec.get("research_interests_standardized_en") or []
    if not kws:
        continue
    for kw in kws:
        l1_people[kw2l1[kw]].add(rec["faculty_id"])

coverage = (
    pd.DataFrame(
        {
            "parent_l1": list(l1_people),
            "n_faculty": [len(v) for v in l1_people.values()],
        }
    )
    .assign(coverage_pct=lambda d: (100 * d["n_faculty"] / N_WITH_KEYWORDS).round(1))
    .sort_values("n_faculty", ascending=False)
    .reset_index(drop=True)
)
over_50 = coverage[coverage["coverage_pct"] > 50]
checks.append(("無任一 L1 在全體覆蓋率 > 50%", over_50.empty, f"{over_50.to_dict('records')}"))

print(coverage.to_string(index=False))
print()
for label, ok, detail in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {label}" + (f"  {detail}" if not ok else ""))

assert all(ok for _, ok, _ in checks), "taxonomy 驗證未通過"

                                    parent_l1  n_faculty  coverage_pct
                        AI & Machine Learning        210          31.7
Vision, Multimedia & Human-Centered Computing        145          21.9
             Systems, Software & Architecture        120          18.1
        Data, Databases & Knowledge Discovery        105          15.9
                    Networks & Communications         96          14.5
              Biomedical & Health Informatics         76          11.5
                 Financial Engineering & Risk         66          10.0
            Asset Pricing & Financial Markets         65           9.8
                           Economics & Policy         65           9.8
             Security, Privacy & Cryptography         64           9.7
    Applied Mathematics & Operations Research         61           9.2
             Statistics & Statistical Methods         60           9.1
                          Theory & Algorithms         56           8.5
      

In [6]:
l2_summary = (
    taxonomy_df.groupby(["parent_l1", "parent_l2"])
    .agg(n_keywords=("keyword_en", "size"), faculty_mentions=("faculty_count", "sum"))
    .reset_index()
)
print(f"L2 總數 = {len(l2_summary)}")
print(l2_summary.to_string(index=False))

L2 總數 = 46
                                    parent_l1                                      parent_l2  n_keywords  faculty_mentions
                        AI & Machine Learning    Artificial Intelligence & Generative Models           3               120
                        AI & Machine Learning               Machine Learning & Deep Learning           8               180
                        AI & Machine Learning           Natural Language & Speech Processing           4                32
    Applied Mathematics & Operations Research                 Game Theory & Mechanism Design           2                15
    Applied Mathematics & Operations Research   Mathematical Analysis & Scientific Computing           5                15
    Applied Mathematics & Operations Research             Optimization & Operations Research           4                41
            Asset Pricing & Financial Markets                    Asset Pricing & Investments           5                47
     

In [7]:
OUT_PATH = OUT_DIR / "research_taxonomy.csv"
taxonomy_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"written -> {OUT_PATH}  ({len(taxonomy_df)} rows)")

written -> <project-root>\data\network-analysis\research_taxonomy.csv  (170 rows)
